In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from libauc.losses import APLoss
from libauc.optimizers import SOAP
from libauc.sampler import DualSampler


def find_cellmob_root() -> Path:
    current = Path(__file__).resolve().parent

    for candidate in [current] + list(current.parents):
        if candidate.name == "CellMob" and (candidate / "Data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the 'CellMob' root folder containing the 'Data' directory. "
        "Make sure this script is stored somewhere inside the CellMob project."
    )


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 5
MAX_WINDOW_SPAN_SECONDS = 3.5

BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 1

NUM_CE_EPOCHS = 15
NUM_SOAP_EPOCHS = 15

CE_LR = 1e-3
SOAP_LR = 1e-3
WEIGHT_DECAY = 1e-5

SOAP_MARGIN = 1.0
SOAP_GAMMA = 0.9
SOAP_SAMPLING_RATE = 0.5

CELLMOB_ROOT = find_cellmob_root()
SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = CELLMOB_ROOT / "Data"
KAUST_6400_DIR = DATA_DIR / "6400 KAUST"

OUTPUT_DIR = SCRIPT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COLUMN = "time"
FEATURE_COLUMNS = [
    "rsrp1", "rsrp2", "rsrp3", "rsrp4",
    "rssi1", "rssi2", "rssi3", "rssi4",
    "rsrq1", "rsrq2", "rsrq3", "rsrq4",
]

LABEL_MAP = {
    "walk": 0,
    "bus": 1,
}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

TRAIN_FILES = {
    "walk": KAUST_6400_DIR / "walk_train_kaust_standardized.csv",
    "bus": KAUST_6400_DIR / "bus_train_kaust_standardized.csv",
}

TEST_FILES = {
    "walk": KAUST_6400_DIR / "walk_test_kaust_standardized_6400windows.csv",
    "bus": KAUST_6400_DIR / "bus_test_kaust_standardized_6400windows.csv",
}

CE_MODEL_PATH = OUTPUT_DIR / "rnn_walk_vs_bus_ce_pretrain.pth"
SOAP_MODEL_PATH = OUTPUT_DIR / "rnn_walk_vs_bus_soap_finetuned.pth"
PR_PLOT_PATH = OUTPUT_DIR / "pr_curve_walk_vs_bus_soap.png"
CM_PLOT_PATH = OUTPUT_DIR / "cm_walk_vs_bus_soap.png"


def time_to_seconds(t) -> float:
    s = str(t).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError(f"Invalid time value: {t!r}")

    parts = s.split(":")
    if len(parts) != 3:
        raise ValueError(f"Bad time format: {t!r}")

    hh = int(parts[0])
    mm = int(parts[1])
    sec_part = parts[2]

    if "." in sec_part:
        ss_str, frac_str = sec_part.split(".", 1)
        ss = int(ss_str)
        micro = int(frac_str.ljust(6, "0")[:6])
    else:
        ss = int(sec_part)
        micro = 0

    return hh * 3600 + mm * 60 + ss + micro / 1_000_000


def build_windows_from_file(csv_path: Path, class_name: str):
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]

    required_cols = [TIME_COLUMN] + FEATURE_COLUMNS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name} is missing columns: {missing}")

    df = df.dropna(subset=required_cols).reset_index(drop=True)
    df[TIME_COLUMN] = df[TIME_COLUMN].astype(str).str.strip()
    df["time_seconds"] = df[TIME_COLUMN].apply(time_to_seconds)

    features = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    times = df["time_seconds"].to_numpy(dtype=np.float64)

    X = []
    y = []

    total_candidate_windows = 0
    kept_windows = 0
    rejected_windows = 0

    for i in range(len(df) - WINDOW_SIZE + 1):
        total_candidate_windows += 1
        span = times[i + WINDOW_SIZE - 1] - times[i]

        if 0 <= span <= MAX_WINDOW_SPAN_SECONDS:
            X.append(features[i:i + WINDOW_SIZE])
            y.append(LABEL_MAP[class_name])
            kept_windows += 1
        else:
            rejected_windows += 1

    if len(X) == 0:
        X = np.empty((0, WINDOW_SIZE, len(FEATURE_COLUMNS)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)
    else:
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.int64)

    stats = {
        "file": csv_path.name,
        "class_name": class_name,
        "rows": len(df),
        "candidate_windows": total_candidate_windows,
        "kept_windows": kept_windows,
        "rejected_windows": rejected_windows,
    }
    return X, y, stats


def build_dataset(file_dict):
    X_all = []
    y_all = []

    for class_name, path in file_dict.items():
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")

        X, y, stats = build_windows_from_file(path, class_name)
        X_all.append(X)
        y_all.append(y)

        print(
            f"{path.name} | rows={stats['rows']} | "
            f"candidate_windows={stats['candidate_windows']} | "
            f"kept={stats['kept_windows']} | rejected={stats['rejected_windows']}"
        )

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    return X_all, y_all


class IndexedSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.targets = self.y.clone()
        self.indices = torch.arange(len(y), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.indices[idx]


class BinaryRNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden).squeeze(1)


def train_one_epoch_ce(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch, _ in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).float()

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


def train_one_epoch_soap(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    total_seen = 0

    for X_batch, y_batch, index_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        index_batch = index_batch.to(DEVICE)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        loss = loss_fn(probs, y_batch, index_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = X_batch.size(0)
        total_loss += loss.item() * bs
        total_seen += bs

    return total_loss / total_seen


@torch.no_grad()
def evaluate_binary(model, loader, threshold=0.5):
    model.eval()

    y_true_all = []
    y_prob_all = []

    for X_batch, y_batch, _ in loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        y_prob_all.append(probs)
        y_true_all.append(y_batch.numpy())

    y_true = np.concatenate(y_true_all).astype(int)
    y_prob = np.concatenate(y_prob_all)

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    ap_bus = average_precision_score(y_true, y_prob)

    y_true_walk = 1 - y_true
    y_prob_walk = 1.0 - y_prob
    ap_walk = average_precision_score(y_true_walk, y_prob_walk)

    macro_ap_binary = (ap_bus + ap_walk) / 2.0

    precision_bus, recall_bus, _ = precision_recall_curve(y_true, y_prob)
    precision_walk, recall_walk, _ = precision_recall_curve(y_true_walk, y_prob_walk)

    return {
        "accuracy": float(acc),
        "confusion_matrix": cm,
        "ap_bus": float(ap_bus),
        "ap_walk": float(ap_walk),
        "macro_ap_binary": float(macro_ap_binary),
        "precision_bus": precision_bus,
        "recall_bus": recall_bus,
        "precision_walk": precision_walk,
        "recall_walk": recall_walk,
        "y_true": y_true,
        "y_prob": y_prob,
    }


def plot_pr_curves(results, save_path: Path):
    plt.figure(figsize=(8, 6))

    plt.plot(
        results["recall_bus"],
        results["precision_bus"],
        label=f"bus positive (AP={results['ap_bus']:.4f})"
    )
    plt.plot(
        results["recall_walk"],
        results["precision_walk"],
        label=f"walk positive (AP={results['ap_walk']:.4f})"
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("PR Curves - Walk vs Bus (SOAP)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_confusion_matrix(cm, save_path: Path):
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["walk", "bus"]
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix - Walk vs Bus")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def print_class_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{name} class distribution:")
    for u, c in zip(unique, counts):
        print(f"  {IDX_TO_LABEL[int(u)]}: {int(c)}")


def main():
    if not KAUST_6400_DIR.exists():
        raise FileNotFoundError(f"Data directory does not exist: {KAUST_6400_DIR}")

    print(f"CellMob root: {CELLMOB_ROOT}")
    print(f"6400 KAUST directory: {KAUST_6400_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")

    print("Building walk/bus train dataset...")
    X_train, y_train = build_dataset(TRAIN_FILES)

    print("\nBuilding walk/bus test dataset...")
    X_test, y_test = build_dataset(TEST_FILES)

    print("\nFinal shapes:")
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test :", X_test.shape)
    print("y_test :", y_test.shape)

    print_class_distribution("Train", y_train)
    print_class_distribution("Test", y_test)

    train_dataset = IndexedSequenceDataset(X_train, y_train)
    test_dataset = IndexedSequenceDataset(X_test, y_test)

    ce_train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    sampler = DualSampler(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampling_rate=SOAP_SAMPLING_RATE
    )
    soap_train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model = BinaryRNNClassifier(
        input_size=len(FEATURE_COLUMNS),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
    ).to(DEVICE)

    ce_criterion = nn.BCEWithLogitsLoss()
    ce_optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CE_LR,
        weight_decay=WEIGHT_DECAY
    )

    print(f"\n[Stage 1] CE pretraining on {DEVICE}...\n")
    for epoch in range(1, NUM_CE_EPOCHS + 1):
        train_loss = train_one_epoch_ce(model, ce_train_loader, ce_optimizer, ce_criterion)
        print(f"CE Epoch {epoch:02d}/{NUM_CE_EPOCHS} - Train Loss: {train_loss:.6f}")

    torch.save(model.state_dict(), CE_MODEL_PATH)
    print(f"\nSaved CE pretrained model to: {CE_MODEL_PATH}")

    ce_results = evaluate_binary(model, test_loader)
    print("\n==================== CE PRETRAIN RESULTS ====================")
    print(f"AP / AUPRC (bus positive):  {ce_results['ap_bus']:.6f}")
    print(f"AP / AUPRC (walk positive): {ce_results['ap_walk']:.6f}")
    print(f"Binary Macro AP:            {ce_results['macro_ap_binary']:.6f}")
    print(f"Accuracy:                   {ce_results['accuracy']:.6f}")
    print("Confusion Matrix [rows=true, cols=pred]:")
    print(ce_results["confusion_matrix"])

    soap_model = BinaryRNNClassifier(
        input_size=len(FEATURE_COLUMNS),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
    ).to(DEVICE)

    soap_model.load_state_dict(torch.load(CE_MODEL_PATH, map_location=DEVICE))
    soap_model.fc.reset_parameters()

    loss_fn = APLoss(
        data_len=len(train_dataset),
        gamma=SOAP_GAMMA,
        margin=SOAP_MARGIN,
        surr_loss="squared_hinge",
    )

    soap_optimizer = SOAP(
        soap_model.parameters(),
        lr=SOAP_LR,
        mode="adam",
        weight_decay=WEIGHT_DECAY,
    )

    print(f"\n[Stage 2] SOAP fine-tuning on {DEVICE}...\n")
    for epoch in range(1, NUM_SOAP_EPOCHS + 1):
        train_loss = train_one_epoch_soap(soap_model, soap_train_loader, soap_optimizer, loss_fn)
        test_results = evaluate_binary(soap_model, test_loader)

        print(
            f"SOAP Epoch {epoch:02d}/{NUM_SOAP_EPOCHS} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Test AP(bus): {test_results['ap_bus']:.6f} | "
            f"Test Macro AP: {test_results['macro_ap_binary']:.6f}"
        )

    final_results = evaluate_binary(soap_model, test_loader)

    print("\n==================== SOAP FINAL RESULTS ====================")
    print(f"Primary paper-aligned AP / AUPRC (bus positive): {final_results['ap_bus']:.6f}")
    print(f"Symmetric AP / AUPRC (walk positive):            {final_results['ap_walk']:.6f}")
    print(f"Binary Macro AP:                                 {final_results['macro_ap_binary']:.6f}")
    print(f"Accuracy:                                        {final_results['accuracy']:.6f}")

    print("\nConfusion Matrix [rows=true, cols=pred]:")
    print(final_results["confusion_matrix"])

    plot_pr_curves(final_results, PR_PLOT_PATH)
    print(f"\nPR curves saved as: {PR_PLOT_PATH}")

    plot_confusion_matrix(final_results["confusion_matrix"], CM_PLOT_PATH)
    print(f"Confusion matrix plot saved as: {CM_PLOT_PATH}")

    torch.save(soap_model.state_dict(), SOAP_MODEL_PATH)
    print(f"SOAP model saved as: {SOAP_MODEL_PATH}")


if __name__ == "__main__":
    main()

Building walk/bus train dataset...
walk_train_kaust_standardized.csv | rows=282015 | candidate_windows=282011 | kept=281616 | rejected=395
bus_train_kaust_standardized.csv | rows=25783 | candidate_windows=25779 | kept=25751 | rejected=28

Building walk/bus test dataset...
walk_test_kaust_standardized_6400windows.csv | rows=6408 | candidate_windows=6404 | kept=6400 | rejected=4
bus_test_kaust_standardized_6400windows.csv | rows=6412 | candidate_windows=6408 | kept=6400 | rejected=8

Final shapes:
X_train: (307367, 5, 12)
y_train: (307367,)
X_test : (12800, 5, 12)
y_test : (12800,)

Train class distribution:
  walk: 281616
  bus: 25751

Test class distribution:
  walk: 6400
  bus: 6400

[Stage 1] CE pretraining on cpu...

CE Epoch 01/15 - Train Loss: 0.161917
CE Epoch 02/15 - Train Loss: 0.032268
CE Epoch 03/15 - Train Loss: 0.022353
CE Epoch 04/15 - Train Loss: 0.019256
CE Epoch 05/15 - Train Loss: 0.016750
CE Epoch 06/15 - Train Loss: 0.014981
CE Epoch 07/15 - Train Loss: 0.014960
CE E

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    average_precision_score,
    accuracy_score,
    confusion_matrix,
)


def find_cellmob_root() -> Path:
    current = Path(__file__).resolve().parent

    for candidate in [current] + list(current.parents):
        if candidate.name == "CellMob" and (candidate / "Data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the 'CellMob' root folder containing the 'Data' directory. "
        "Make sure this script is stored somewhere inside the CellMob project."
    )


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 5
MAX_WINDOW_SPAN_SECONDS = 3.5

BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 1
NUM_EPOCHS = 15

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

CELLMOB_ROOT = find_cellmob_root()
SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = CELLMOB_ROOT / "Data"
KAUST_6400_DIR = DATA_DIR / "6400 KAUST"

OUTPUT_DIR = SCRIPT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COLUMN = "time"
FEATURE_COLUMNS = [
    "rsrp1", "rsrp2", "rsrp3", "rsrp4",
    "rssi1", "rssi2", "rssi3", "rssi4",
    "rsrq1", "rsrq2", "rsrq3", "rsrq4",
]

LABEL_MAP = {
    "walk": 0,
    "bus": 1,
}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

TRAIN_FILES = {
    "walk": KAUST_6400_DIR / "walk_train_kaust_standardized.csv",
    "bus": KAUST_6400_DIR / "bus_train_kaust_standardized.csv",
}

TEST_FILES = {
    "walk": KAUST_6400_DIR / "walk_test_kaust_standardized_6400windows.csv",
    "bus": KAUST_6400_DIR / "bus_test_kaust_standardized_6400windows.csv",
}

MODEL_PATH = OUTPUT_DIR / "rnn_walk_vs_bus_ce_from_scratch.pth"


def time_to_seconds(t) -> float:
    s = str(t).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError(f"Invalid time value: {t!r}")

    parts = s.split(":")
    if len(parts) != 3:
        raise ValueError(f"Bad time format: {t!r}")

    hh = int(parts[0])
    mm = int(parts[1])
    sec_part = parts[2]

    if "." in sec_part:
        ss_str, frac_str = sec_part.split(".", 1)
        ss = int(ss_str)
        micro = int(frac_str.ljust(6, "0")[:6])
    else:
        ss = int(sec_part)
        micro = 0

    return hh * 3600 + mm * 60 + ss + micro / 1_000_000


def build_windows_from_file(csv_path: Path, class_name: str):
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]

    required_cols = [TIME_COLUMN] + FEATURE_COLUMNS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name} is missing columns: {missing}")

    df = df.dropna(subset=required_cols).reset_index(drop=True)
    df[TIME_COLUMN] = df[TIME_COLUMN].astype(str).str.strip()
    df["time_seconds"] = df[TIME_COLUMN].apply(time_to_seconds)

    features = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    times = df["time_seconds"].to_numpy(dtype=np.float64)

    X = []
    y = []

    total_candidate_windows = 0
    kept_windows = 0
    rejected_windows = 0

    for i in range(len(df) - WINDOW_SIZE + 1):
        total_candidate_windows += 1
        span = times[i + WINDOW_SIZE - 1] - times[i]

        if 0 <= span <= MAX_WINDOW_SPAN_SECONDS:
            X.append(features[i:i + WINDOW_SIZE])
            y.append(LABEL_MAP[class_name])
            kept_windows += 1
        else:
            rejected_windows += 1

    if len(X) == 0:
        X = np.empty((0, WINDOW_SIZE, len(FEATURE_COLUMNS)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)
    else:
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.int64)

    stats = {
        "file": csv_path.name,
        "class_name": class_name,
        "rows": len(df),
        "candidate_windows": total_candidate_windows,
        "kept_windows": kept_windows,
        "rejected_windows": rejected_windows,
    }

    return X, y, stats


def build_dataset(file_dict):
    X_all = []
    y_all = []

    for class_name, path in file_dict.items():
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")

        X, y, stats = build_windows_from_file(path, class_name)
        X_all.append(X)
        y_all.append(y)

        print(
            f"{path.name} | rows={stats['rows']} | "
            f"candidate_windows={stats['candidate_windows']} | "
            f"kept={stats['kept_windows']} | rejected={stats['rejected_windows']}"
        )

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    return X_all, y_all


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BinaryRNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden).squeeze(1)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).float()

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()

    all_probs = []
    all_true = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_true.append(y_batch.numpy())

    y_true = np.concatenate(all_true).astype(int)
    y_prob = np.concatenate(all_probs)
    y_pred = (y_prob >= threshold).astype(int)

    ap_bus = average_precision_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "ap_bus": float(ap_bus),
        "accuracy": float(acc),
        "confusion_matrix": cm,
    }


def print_class_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{name} class distribution:")
    for u, c in zip(unique, counts):
        print(f"  {IDX_TO_LABEL[int(u)]}: {int(c)}")


def main():
    if not KAUST_6400_DIR.exists():
        raise FileNotFoundError(f"Data directory does not exist: {KAUST_6400_DIR}")

    print(f"CellMob root: {CELLMOB_ROOT}")
    print(f"6400 KAUST directory: {KAUST_6400_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")

    print("Building walk/bus train dataset...")
    X_train, y_train = build_dataset(TRAIN_FILES)

    print("\nBuilding walk/bus test dataset...")
    X_test, y_test = build_dataset(TEST_FILES)

    print("\nFinal shapes:")
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test :", X_test.shape)
    print("y_test :", y_test.shape)

    print_class_distribution("Train", y_train)
    print_class_distribution("Test", y_test)

    train_dataset = SequenceDataset(X_train, y_train)
    test_dataset = SequenceDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = BinaryRNNClassifier(
        input_size=len(FEATURE_COLUMNS),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    print(f"\nTraining Cross-Entropy from scratch on {DEVICE}...\n")
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        results = evaluate(model, test_loader)

        print(
            f"CE Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Test AP(bus): {results['ap_bus']:.6f} | "
            f"Test Acc: {results['accuracy']:.6f}"
        )

    final_results = evaluate(model, test_loader)

    print("\n==================== CE RESULTS ====================")
    print(f"AP / AUPRC (bus positive): {final_results['ap_bus']:.6f}")
    print(f"Accuracy:                  {final_results['accuracy']:.6f}")
    print("Confusion Matrix [rows=true, cols=pred]:")
    print(final_results["confusion_matrix"])

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"\nSaved CE model to: {MODEL_PATH}")


if __name__ == "__main__":
    main()

Building walk/bus train dataset...
walk_train_kaust_standardized.csv | rows=282015 | candidate_windows=282011 | kept=281616 | rejected=395
bus_train_kaust_standardized.csv | rows=25783 | candidate_windows=25779 | kept=25751 | rejected=28

Building walk/bus test dataset...
walk_test_kaust_standardized_6400windows.csv | rows=6408 | candidate_windows=6404 | kept=6400 | rejected=4
bus_test_kaust_standardized_6400windows.csv | rows=6412 | candidate_windows=6408 | kept=6400 | rejected=8

Final shapes:
X_train: (307367, 5, 12)
y_train: (307367,)
X_test : (12800, 5, 12)
y_test : (12800,)

Train class distribution:
  walk: 281616
  bus: 25751

Test class distribution:
  walk: 6400
  bus: 6400

Training Cross-Entropy from scratch on cpu...

CE Epoch 01/15 | Train Loss: 0.161917 | Test AP(bus): 0.998384 | Test Acc: 0.971953
CE Epoch 02/15 | Train Loss: 0.031748 | Test AP(bus): 0.999407 | Test Acc: 0.975625
CE Epoch 03/15 | Train Loss: 0.023197 | Test AP(bus): 0.999700 | Test Acc: 0.980391
CE Epoc